In [190]:
from scenario import Scenario
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from numpy.typing import NDArray
from libraries import ACTIONS

In [191]:
scene = Scenario()
rnd_policy = np.ones((scene.states.n, scene.actions.n)) / scene.actions.n

$$
q_{\pi}(s, a)=\sum_{s', r} p(s', r | s, a)[r+\gamma v_{\pi}(s')]
$$

In [192]:
EPSILON = 1e-100
GAMMA = 1-EPSILON

In [193]:
def state_quality(environ:Scenario, vector:NDArray, state:int, gamma:int=1-EPSILON):
    quality = np.zeros(environ.actions.n)
    # print(f'{quality}')
    
    for action in range(environ.actions.n):
        for prob, next_state, reward, _ in environ.transition[state][action]:
            
            cell = prob * (reward + gamma*vector[next_state])
            quality[action] += cell
            # print(f'act {action}: {round(cell,4), quality}')

    return quality

In [194]:
starting_policy = [0.0139398,  0.01163093, 0.02095299, 0.01047649,
                   0.01624867, 0.0,        0.04075154, 0.0       ,
                   0.0348062,  0.08816993, 0.14205316, 0.0       ,
                   0.0,        0.17582037, 0.43929118, 0.0       ]
empty_policy = np.zeros((scene.states.n, scene.actions.n))
print(empty_policy)

for state in range(scene.states.n):
    empty_policy[state] = state_quality(scene, starting_policy, state)

print(empty_policy)



[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[0.01470942 0.0139398  0.0139398  0.01317018]
 [0.00852358 0.01163093 0.01086131 0.01550791]
 [0.02444515 0.02095299 0.02406034 0.01435347]
 [0.01047649 0.01047649 0.00698433 0.01396866]
 [0.02166489 0.01701829 0.01624867 0.01006282]
 [0.         0.         0.         0.        ]
 [0.05433538 0.04735105 0.05433538 0.00698433]
 [0.         0.         0.         0.        ]
 [0.01701829 0.04099204 0.0348062  0.04640827]
 [0.07020886 0.11755991 0.10595784 0.05895312]
 [0.18940422 0.17582037 0.16001424 0.04297382]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.08799677 0.20503718 0.23442716 0.17582037]
 [0.25238824 0.53837052 0.52711478 0.43929118]
 [0.         0.         0.         0.        ]]


In [195]:
def mejorar_politica(environ:Scenario, vector:NDArray, gamma:float=1-EPSILON):
    new_policy = np.zeros((environ.states.n, environ.actions.n))
    new_arrows = []
    
    for state in range(scene.states.n):
        quality = state_quality(scene, vector, state, gamma)
        new_policy[state][np.argmax(quality)] = 1
        new_arrows.append(ACTIONS[np.argmax(quality)])

    return new_policy, new_arrows

In [196]:
better_policy, arrows = mejorar_politica(scene, starting_policy)
print(better_policy)
print(arrows)


[[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]
['←', '↑', '←', '↑', '←', '←', '←', '←', '↑', '↓', '←', '←', '←', '→', '↓', '←']


In [197]:
EPSILON = 1e-100
GAMMA = 1-EPSILON

In [198]:
def evaluar_politica(env: Scenario, policy, gamma=GAMMA, epsilon=EPSILON):
    delta: float = 1
    deltas = [delta]
    power = 0

    # flechas (soluciones)
    state_values = np.zeros(env.states.n)
    spaces_vector = [state_values.copy()]  # Copia inicial
    # print(f'{policy_vector=}')

    while delta > epsilon:
        delta = 0  # Reiniciar delta para esta iteración

        # Iteramos sobre el espacio de estados:
        for state in range(env.states.n):
            cell: float = 0

            # Iteramos sobre las diferentes acciones realizables
            for action, policy_prob in enumerate(policy[state]):
                for prob, next_state, reward, _ in env.transition[state][action]:
                    cell += (
                        policy_prob * prob *
                        (reward + (gamma) * state_values[next_state])
                    )

            # Calcular el cambio máximo
            delta = max(delta, np.abs(state_values[state] - cell))
            deltas.append(delta)
            state_values[state] = cell

        # Guardar una copia del vector después de esta iteración
        power += 1
        spaces_vector.append(state_values.copy())

    return state_values, spaces_vector, deltas

In [211]:
def policy_iteration(environ:Scenario, gamma=1-EPSILON, epsilon=EPSILON):
    actual_policy = np.ones((environ.states.n, environ.actions.n)) / environ.actions.n
    delta:int = 1
    evaluation = None
    
    while delta > epsilon:
        evaluation, all_evals, deltas = evaluar_politica(environ, actual_policy)
        new_policy, arrows = mejorar_politica(environ, evaluation)
        print(new_policy)
                
        if (new_policy == actual_policy).all():
            break

        actual_policy = np.copy(new_policy)

    print(np.array(arrows).reshape((4,4)))
    
    
    return actual_policy, evaluation

In [212]:
iterated_policy, policy_evaluation = policy_iteration(scene)

[[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]
[[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]
[[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]
[['←' '↑' '↑' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]


In [213]:
# print(iterated_policy.reshape((4, 4)))
# print(policy_evaluation.reshape((4, 4)))